# 03 - İmalat Proses Parametre Optimizasyonu

Bu notebook, Gaussian Process tabanlı Bayes optimizasyonunun imalat/proses mühendisliği bağlamında nasıl kullanılabileceğini gösterir.

Senaryo:

Bir CNC operasyonunda üç proses parametresi ayarlanacaktır:

- iş mili devri
- ilerleme hızı
- kesme derinliği

Gerçek bir fabrikada amaç fonksiyonu:

- fiziksel deney,
- sensör ölçümü,
- kalite laboratuvarı,
- dijital ikiz,
- FEM/CFD benzeri pahalı analiz

üzerinden gelebilir.

Bu notebook'taki amaç fonksiyonu ise tamamen **sentetik ve eğitim amaçlıdır**. Fiziksel talaş kaldırma yasası veya doğrulanmış CNC modeli değildir.

## 1. Neden Bayes optimizasyonu?

Tam faktöriyel deney tasarımı üç parametrede bile hızla büyüyebilir.

Örneğin her parametre için 20 seviye denenirse:

\[
20^3 = 8000
\]

deney gerekir.

Bir fiziksel deney 15 dakika sürüyorsa bu yaklaşım pratik olmayabilir.

Bayes optimizasyonu, az sayıda başlangıç deneyinden sonra GP surrogate modelini kullanarak sonraki deneyleri bilgi değeri yüksek bölgelere yönlendirmeye çalışır.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning

PROJE_KOKU = Path.cwd()
if not (PROJE_KOKU / "src").exists() and (PROJE_KOKU.parent / "src").exists():
    PROJE_KOKU = PROJE_KOKU.parent

if not (PROJE_KOKU / "src").exists():
    raise FileNotFoundError(
        "Bu notebook'u repository kök dizininden çalıştırın."
    )

sys.path.insert(0, str((PROJE_KOKU / "src").resolve()))

from gaussian_bo import GaussianProcessBayesOptimizer

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. Karar değişkenleri

Arama aralıkları:

| Parametre | Alt sınır | Üst sınır |
|---|---:|---:|
| İş mili devri | 1500 rpm | 4500 rpm |
| İlerleme hızı | 100 mm/dk | 500 mm/dk |
| Kesme derinliği | 0.5 mm | 3.0 mm |

Bu değişkenlerin ölçekleri oldukça farklıdır. Repository'deki optimizer GP'ye vermeden önce tüm karar değişkenlerini `[0, 1]` aralığına ölçekler.

## 3. Sentetik black-box amaç fonksiyonu

Aşağıdaki fonksiyon birkaç kavramı tek bir eğitim amaçlı maliyet puanında birleştirir:

- hedef bölgeden uzaklaşma cezası
- parametre etkileşimleri
- verimlilik cezası
- enerji benzeri ceza
- hafif nonconvex yapı

Buradaki katsayılar gerçek üretim verisi değildir.

Gerçek projede bu fonksiyonun yerine doğrudan:

```python
def amac_fonksiyonu(x):
    sonuc = fiziksel_deneyi_calistir(x)
    return sonuc.toplam_maliyet
```

veya simülasyon/ölçüm arayüzü konur.

In [ ]:
def cnc_sentetik_amac(x):
    devir, ilerleme, derinlik = map(float, x)

    # Değişkenleri eğitim amaçlı sentetik formülde normalize ediyoruz.
    u = (devir - 1500.0) / (4500.0 - 1500.0)
    v = (ilerleme - 100.0) / (500.0 - 100.0)
    w = (derinlik - 0.5) / (3.0 - 0.5)

    kalite_kaybi = (
        85.0 * (u - 0.62) ** 2
        + 110.0 * (v - 0.43) ** 2
        + 70.0 * (w - 0.55) ** 2
    )

    etkilesim_kaybi = (
        22.0 * (u - v) ** 2
        + 10.0 * (v - w) ** 2
    )

    verimlilik_kaybi = (
        18.0 / (0.25 + v)
        + 8.0 / (0.25 + w)
    )

    enerji_benzeri_kayip = 8.0 * u**2 + 5.0 * w**2

    nonconvex_bilesen = 2.5 * np.sin(6.0 * u) * np.cos(4.0 * v)

    return (
        kalite_kaybi
        + etkilesim_kaybi
        + verimlilik_kaybi
        + enerji_benzeri_kayip
        + nonconvex_bilesen
    )

## 4. Bayes optimizasyonunu çalıştırma

Toplam gerçek amaç fonksiyonu değerlendirme sayısı:

```text
8 başlangıç + 25 sequential iterasyon = 33 değerlendirme
```

olacaktır.

Bu, 8000 noktalı örnek tam faktöriyel tasarıma göre çok daha küçük bir deney bütçesidir. Ancak bulunan çözümün global optimum olduğu garanti edilmez.

In [ ]:
optimizer = GaussianProcessBayesOptimizer(
    amac_fonksiyonu=cnc_sentetik_amac,
    sinirlar=[
        [1500.0, 4500.0],
        [100.0, 500.0],
        [0.5, 3.0],
    ],
    baslangic_noktasi_sayisi=8,
    edinim_fonksiyonu="ei",
    xi=0.01,
    random_state=42,
)

sonuc = optimizer.optimize_et(
    iterasyon_sayisi=25,
    ayrintili=True,
)

devir, ilerleme, derinlik = sonuc.en_iyi_x

print("En iyi bulunan parametreler:")
print(f"İş mili devri: {devir:.1f} rpm")
print(f"İlerleme hızı: {ilerleme:.1f} mm/dk")
print(f"Kesme derinliği: {derinlik:.3f} mm")
print(f"Sentetik amaç değeri: {sonuc.en_iyi_y:.4f}")

## 5. Yakınsama grafiği

In [ ]:
kumulatif_en_iyi = np.minimum.accumulate(sonuc.y_gozlenen)

plt.figure(figsize=(9, 5))
plt.plot(
    np.arange(1, len(kumulatif_en_iyi) + 1),
    kumulatif_en_iyi,
    marker="o",
)
plt.xlabel("Deney / amaç fonksiyonu değerlendirme sayısı")
plt.ylabel("O ana kadarki en iyi sentetik kayıp")
plt.title("İmalat proses parametre optimizasyonu")
plt.grid(True)
plt.show()

## 6. Gerçek üretim problemine geçerken yapılması gerekenler

Sentetik fonksiyon yerine gerçek süreç kullanıldığında aşağıdaki konular kritik hale gelir.

### Ölçüm tekrarı

Aynı parametre kombinasyonu birkaç kez denenebilir:

```text
aynı devir + ilerleme + derinlik
-> 5 parça üret
-> kalite ölç
-> ortalama ve varyans hesapla
```

### Kısıtlar

Örnek:

- takım sıcaklığı belirli limiti aşmamalı
- yüzey pürüzlülüğü maksimum değeri aşmamalı
- makine gücü kapasite sınırını aşmamalı
- kesme kuvveti güvenli bölge içinde kalmalı

### Çok amaçlı yapı

Gerçek proses çoğu zaman tek amaçlı değildir:

```text
min yüzey pürüzlülüğü
min çevrim süresi
min enerji
max takım ömrü
```

Bu durumda:

- ekonomik eşdeğer maliyet,
- ağırlıklı skalerleştirme,
- kısıtlı optimizasyon,
- Pareto tabanlı çok amaçlı Bayes optimizasyonu

düşünülebilir.

## 7. DOE ile ilişkisi

Bayes optimizasyonu DOE'nin yerine her durumda geçen bir yöntem değildir.

Klasik DOE özellikle:

- faktör etkilerini yorumlamak,
- etkileşimleri istatistiksel olarak analiz etmek,
- deneysel nedensel çıkarım yapmak

için çok değerlidir.

Bayes optimizasyonunun temel hedefi ise çoğunlukla:

```text
mümkün olduğunca az pahalı deneyle iyi/optimum parametre bölgesi bulmak
```

olarak özetlenebilir.

Bu nedenle üretimde DOE ve BO rakip olmak zorunda değildir. Örneğin ilk deney tasarımı DOE/LHS ile hazırlanıp daha sonra sequential BO ile devam edilebilir.

## 8. Endüstri mühendisi açısından raporlama

Bir gerçek proje raporunda en az şu bilgiler bulunmalıdır:

1. Karar değişkenleri ve fiziksel sınırları
2. Amaç fonksiyonunun birimleri
3. Kısıtlar
4. Başlangıç deney tasarımı
5. GP kernel seçimi
6. Gürültü modeli
7. Acquisition function
8. Toplam deney bütçesi
9. En iyi bulunan parametreler
10. Bağımsız doğrulama deneyleri
11. Baz yöntem karşılaştırması
12. Global optimum garantisi olmadığına dair açık ifade